# Model Selection

## Learning Objectives
1. Compute AIC and BIC from log-likelihood and parameter count using NumPy.
2. Run nested cross-validation in scikit-learn for unbiased model comparison.
3. Decompose bias and variance using repeated hold-out experiments.
4. Demonstrate the no-free-lunch theorem with multiple classifiers.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    cross_val_score, GridSearchCV, StratifiedKFold, KFold
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification, make_regression

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: AIC / BIC Computation (NumPy)


In [ ]:
def log_likelihood_gaussian(y_true: np.ndarray, y_pred: np.ndarray,
                             sigma: float = None) -> float:
    """Log-likelihood under a Gaussian noise model."""
    n = len(y_true)
    residuals = y_true - y_pred
    if sigma is None:
        sigma = np.std(residuals)
    ll = (-n / 2 * np.log(2 * np.pi * sigma ** 2)
          - (1 / (2 * sigma ** 2)) * np.sum(residuals ** 2))
    return ll


def aic(log_lik: float, k: int) -> float:
    """Akaike Information Criterion: AIC = 2k - 2*log(L)."""
    return 2 * k - 2 * log_lik


def bic(log_lik: float, k: int, n: int) -> float:
    """Bayesian Information Criterion: BIC = k*log(n) - 2*log(L)."""
    return k * np.log(n) - 2 * log_lik


n = 100
x = np.linspace(0, 2 * np.pi, n)
y_true_fn = 2 * np.sin(x) + 0.5 * np.cos(3 * x)
y = y_true_fn + np.random.randn(n) * 0.3

print(f"{'Degree':>7} | {'AIC':>10} | {'BIC':>10} | {'RMSE':>8}")
print("-" * 44)
aic_vals, bic_vals = [], []
for deg in range(1, 9):
    X_poly = np.stack([x ** d for d in range(deg + 1)], axis=1)
    coeffs = np.linalg.lstsq(X_poly, y, rcond=None)[0]
    y_pred = X_poly @ coeffs
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    ll = log_likelihood_gaussian(y, y_pred)
    k = deg + 2
    a = aic(ll, k)
    b = bic(ll, k, n)
    aic_vals.append(a); bic_vals.append(b)
    print(f"{deg:>7d} | {a:>10.2f} | {b:>10.2f} | {rmse:>8.4f}")

print(f"\nBest degree by AIC: {np.argmin(aic_vals) + 1}")
print(f"Best degree by BIC: {np.argmin(bic_vals) + 1}")


## Level 2: Nested Cross-Validation with scikit-learn (Model Comparison)


In [ ]:
X_cls, y_cls = make_classification(n_samples=800, n_features=20, n_informative=10,
                                   n_redundant=5, random_state=42)

candidate_models = {
    "LogisticRegression": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=500))]),
        {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {"n_estimators": [50, 100], "max_depth": [3, 5, None]},
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=42),
        {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1]},
    ),
}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

print(f"{'Model':>20} | {'Mean CV Acc':>11} | {'Std':>6}")
print("-" * 44)
nested_scores = {}
for name, (estimator, param_grid) in candidate_models.items():
    try:
        gs = GridSearchCV(estimator, param_grid, cv=inner_cv,
                          scoring='accuracy', n_jobs=-1)
        scores = cross_val_score(gs, X_cls, y_cls, cv=outer_cv,
                                 scoring='accuracy')
    except Exception as exc:
        print(f"Error for {name}: {exc}"); continue
    nested_scores[name] = scores
    print(f"{name:>20} | {scores.mean():>11.4f} | {scores.std():>6.4f}")

best_model = max(nested_scores, key=lambda k: nested_scores[k].mean())
print(f"\nBest model (nested CV): {best_model}")
# Note: for large models/datasets, wrap training in try/except RuntimeError
# to catch out of memory errors: reduce n_estimators or n_samples if OOM occurs.


## Real-World Example 1: Unbiased Nested CV Prevents Overfitting Estimate


In [ ]:
X_n, y_n = make_classification(n_samples=600, n_features=30, n_informative=8,
                                random_state=0)
est = Pipeline([("sc", StandardScaler()),
                ("clf", LogisticRegression(max_iter=500))])
param_grid_n = {"clf__C": [0.001, 0.01, 0.1, 1, 10]}

# Simple (non-nested) CV: select best C on same data
gs_simple = GridSearchCV(est, param_grid_n, cv=5, scoring="accuracy")
gs_simple.fit(X_n, y_n)
simple_score = gs_simple.best_score_

# Nested CV: inner CV for selection, outer for unbiased eval
outer_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
gs_nested = GridSearchCV(est, param_grid_n, cv=inner_kf, scoring="accuracy")
nested_scores_n = cross_val_score(gs_nested, X_n, y_n, cv=outer_kf,
                                   scoring="accuracy")
nested_mean = nested_scores_n.mean()

print(f"Simple CV best accuracy  : {simple_score:.4f}  <- optimistic (data leakage)")
print(f"Nested CV accuracy       : {nested_mean:.4f}  +/- {nested_scores_n.std():.4f}")
print(f"Optimism bias            : {simple_score - nested_mean:+.4f}")
print(f"\nConclusion: use nested CV for unbiased model selection.")


## Real-World Example 2: Bias-Variance Decomposition via Repeated Hold-Out


In [ ]:
def bias_variance_decompose(model_factory, X_all, y_all, n_trials=50, test_size=0.2):
    """Estimate bias^2 and variance via repeated train/test splits."""
    n_test = int(len(X_all) * test_size)
    all_preds = []
    rng = np.random.default_rng(42)

    for _ in range(n_trials):
        idx = rng.permutation(len(X_all))
        X_tr, y_tr = X_all[idx[n_test:]], y_all[idx[n_test:]]
        X_te, y_te_common = X_all[idx[:n_test]], y_all[idx[:n_test]]
        model = model_factory()
        model.fit(X_tr, y_tr)
        all_preds.append(model.predict(X_te))

    preds = np.array(all_preds)
    y_te = y_te_common
    mean_pred = preds.mean(axis=0)
    bias2 = np.mean((mean_pred - y_te) ** 2)
    variance = np.mean(preds.var(axis=0))
    return bias2, variance


X_bv, y_bv = make_regression(n_samples=500, n_features=10, noise=0.5, random_state=42)
X_bv = StandardScaler().fit_transform(X_bv)

models_bv = {
    "Ridge(alpha=100)":  lambda: Ridge(alpha=100),
    "Ridge(alpha=1)":    lambda: Ridge(alpha=1),
    "Ridge(alpha=0.01)": lambda: Ridge(alpha=0.01),
    "DecisionTree(d=2)": lambda: DecisionTreeRegressor(max_depth=2),
    "DecisionTree(full)": lambda: DecisionTreeRegressor(),
}

print(f"{'Model':>22} | {'Bias^2':>8} | {'Variance':>9} | {'Total':>8}")
print("-" * 56)
for name, factory in models_bv.items():
    try:
        b2, var = bias_variance_decompose(factory, X_bv, y_bv)
    except Exception:
        continue
    print(f"{name:>22} | {b2:>8.4f} | {var:>9.4f} | {b2+var:>8.4f}")


## Real-World Example 3: No-Free-Lunch Theorem Demonstration


In [ ]:
from sklearn.datasets import make_circles, make_moons

classifiers = {
    "Logistic Reg": Pipeline([("sc", StandardScaler()),
                               ("clf", LogisticRegression(max_iter=500))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42),
    "SVM (RBF)":     Pipeline([("sc", StandardScaler()), ("clf", SVC(kernel='rbf'))]),
    "GaussianNB":    GaussianNB(),
    "KNN (k=5)":     KNeighborsClassifier(n_neighbors=5),
}

datasets = {
    "Linear":  make_classification(n_samples=500, n_features=10, n_informative=5,
                                   random_state=42),
    "Circles": make_circles(n_samples=500, noise=0.1, random_state=42),
    "Moons":   make_moons(n_samples=500, noise=0.15, random_state=42),
    "HighDim": make_classification(n_samples=500, n_features=50, n_informative=5,
                                   n_redundant=20, random_state=42),
}

cv_nfl = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_nfl = {ds: {} for ds in datasets}

for ds_name, (X_ds, y_ds) in datasets.items():
    for clf_name, clf in classifiers.items():
        scores = cross_val_score(clf, X_ds, y_ds, cv=cv_nfl, scoring="accuracy")
        results_nfl[ds_name][clf_name] = scores.mean()

header = f"{'Dataset':<12}" + "".join(f"{n[:9]:>12}" for n in classifiers)
print(header)
print("-" * len(header))
for ds_name, clf_scores in results_nfl.items():
    row = f"{ds_name:<12}" + "".join(f"{v:>12.3f}" for v in clf_scores.values())
    print(row)

best_per_ds = {ds: max(scores, key=scores.get)
               for ds, scores in results_nfl.items()}
print("\nBest classifier per dataset:")
for ds, clf in best_per_ds.items():
    print(f"  {ds}: {clf}")
